## Step 1 — Install dependencies

In [1]:
!pip install pytorch-fid torchmetrics gradio -q

## Step 2 — Clone repo and set path
If you see a "already exists" error on the clone, ignore it and keep going.

In [2]:
!git clone -b Frontend https://github.com/youlkar/damage-diffusion.git

fatal: destination path 'damage-diffusion' already exists and is not an empty directory.


In [3]:
import sys, os
sys.path.insert(0, '/content/damage-diffusion/backend')
os.chdir('/content/damage-diffusion')
os.system('git checkout Frontend')

0

In [ ]:
import gradio as gr
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as transforms

from inference import load_model, generate_from_mask
from utils.metrics import compute_iou, compute_pixel_accuracy, compute_fid_kid_scores
from utils.visualization import denormalize

# globals
model  = None
device = None

def load(checkpoint_path, device_choice):
    global model, device
    device = device_choice
    model, _ = load_model(checkpoint_path, device=device)
    return f"Model loaded from {checkpoint_path} on {device}"

def preprocess_mask(mask_img, image_size=128):
    pil = Image.fromarray(mask_img.astype(np.uint8)).convert("L")
    pil = pil.resize((image_size, image_size))
    t   = transforms.ToTensor()(pil)
    t   = (t > 0.5).float()
    return t.unsqueeze(0)

def tensor_to_pil(t):
    t = t.squeeze(0).cpu()
    t = denormalize(t).clamp(0, 1)
    return transforms.ToPILImage()(t)

def generate(upload, draw, num_steps, num_samples):
    if model is None:
        return None, "Load a model checkpoint first."
    # upload takes priority, fall back to draw
    mask_np = upload if upload is not None else draw
    if mask_np is None:
        return None, "Please upload or draw a mask first."
    mask_tensor = preprocess_mask(mask_np)
    generated   = generate_from_mask(
        model, mask_tensor,
        num_samples=num_samples,
        num_inference_steps=num_steps,
        device=device
    )
    return tensor_to_pil(generated[0]), "Done!"

def evaluate(gen_img, gt_mask):
    if gen_img is None or gt_mask is None:
        return "Please provide both a generated image and a ground truth mask."
    gen_t  = transforms.ToTensor()(Image.fromarray(gen_img)).unsqueeze(0)
    mask_t = preprocess_mask(gt_mask)
    iou    = compute_iou(gen_t, mask_t)
    acc    = compute_pixel_accuracy(gen_t, mask_t)
    return f"IoU:            {iou:.4f}\nPixel Accuracy: {acc:.4f}\n(FID/KID require a batch — run eval.py for full scores)"

# UI
with gr.Blocks(title="DamageDiffusion") as demo:
    gr.Markdown("# DamageDiffusion — Crack Image Generator")

    with gr.Row():
        ckpt_input  = gr.Textbox(label="Checkpoint path", placeholder="checkpoints/best_model.pt")
        device_drop = gr.Dropdown(choices=["cuda", "mps", "cpu"], value="cuda", label="Device")
        load_btn    = gr.Button("Load Model")
    load_status = gr.Textbox(label="Status", interactive=False)
    load_btn.click(load, inputs=[ckpt_input, device_drop], outputs=load_status)

    with gr.Tabs():

        # Generate
        with gr.Tab("Generate"):
            gr.Markdown("**Option 1:** Upload a mask image — OR — **Option 2:** Draw one below. Then hit Generate.")
            with gr.Row():
                with gr.Column():
                    mask_upload = gr.Image(
                        label="Option 1 — Upload mask",
                        type="numpy",
                        sources=["upload"],
                    )
                    gr.Markdown("— OR —")
                    mask_draw = gr.Sketchpad(
                        label="Option 2 — Draw mask",
                        type="numpy",
                        image_mode="RGB",
                        canvas_size=(512, 512),
                    )
                    num_steps   = gr.Slider(10, 1000, value=50, step=10, label="Inference steps")
                    num_samples = gr.Slider(1, 4, value=1, step=1, label="Num samples")
                    gen_btn     = gr.Button("Generate", variant="primary")
                with gr.Column():
                    gen_output = gr.Image(label="Generated image")
                    gen_status = gr.Textbox(label="Status", interactive=False)

            gen_btn.click(
                generate,
                inputs=[mask_upload, mask_draw, num_steps, num_samples],
                outputs=[gen_output, gen_status]
            )

        # Evaluate
        with gr.Tab("Evaluate"):
            gr.Markdown("Compare a generated image against a ground truth mask.")
            with gr.Row():
                eval_gen  = gr.Image(label="Generated image",   type="numpy", sources=["upload"])
                eval_mask = gr.Image(label="Ground truth mask", type="numpy", sources=["upload"])
            eval_btn    = gr.Button("Compute metrics", variant="primary")
            eval_output = gr.Textbox(label="Metrics", interactive=False, lines=4)
            eval_btn.click(evaluate, inputs=[eval_gen, eval_mask], outputs=eval_output)

demo.launch(share=True, debug=True)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a8614f7fbdad78d68b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
